# Clase 10 — Feedback y Auto-corrección · versión **LangChain v1**

**Agentes IA — UTEC** · Dr. Vicente Machaca Arceda

Este notebook cubre lo mismo que `10_feedback.ipynb`, pero **sin LangGraph**: solo LangChain v1.

| | LangGraph (`10_feedback.ipynb`) | LangChain v1 (este) |
|---|---|---|
| Control del loop | `StateGraph` + aristas condicionales | **Python plano** (`while`) |
| Composición | nodos y estado compartido | **LCEL**: `prompt \| model \| parser` |
| Uso de herramientas | nodo de herramientas manual | **`create_agent`** |

> **Por qué importa el contraste:** en LangGraph *declaras* el flujo; en LangChain v1 *compones* piezas y el flujo lo escribes tú. Para loops simples de feedback, lo segundo suele ser más corto y más fácil de depurar.

## Configuración

In [ ]:
%pip install -qU "langchain>=1.0" "langchain[google-genai]"

In [1]:
import os, getpass

from dotenv import load_dotenv
load_dotenv()

# --- API de LangChain v1 ---
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

import langchain
print("langchain", langchain.__version__)

#model = init_chat_model("google_genai:gemini-2.5-flash", temperature=0.9)
model = init_chat_model(
    model="gpt-4.1-nano",  # Puedes usar "gpt-3.5-turbo" para ahorrar costos
    temperature=0.7
)

# OJO v1: .text es una PROPIEDAD, no un metodo (en v0 era .text())
print(model.invoke("Responde solo: OK").text)

langchain 1.3.10
OK


---
# Parte 1 — Self-Refine con LCEL

Tres cadenas independientes, cada una `prompt | model | StrOutputParser()`.
El ciclo lo controla un `while` de Python.

Recuerda de la clase: **no hay verificador externo**. El propio modelo decide cuándo parar.

In [2]:
parser = StrOutputParser()

generar = (
    ChatPromptTemplate.from_template(
        "Resuelve esta tarea de forma concisa:\n\n{tarea}"
    )
    | model
    | parser
)

criticar = (
    ChatPromptTemplate.from_template(
        "Tarea:\n{tarea}\n\n"
        "Respuesta propuesta:\n{borrador}\n\n"
        "Critica la respuesta buscando opciones de mejora en la explicación y fundamentos. "
        "Si no encuentras ningun problema real, responde exactamente: SIN_PROBLEMAS"
    )
    | model
    | parser
)

refinar = (
    ChatPromptTemplate.from_template(
        "Tarea:\n{tarea}\n\n"
        "Respuesta anterior:\n{borrador}\n\n"
        "Critica recibida:\n{critica}\n\n"
        "Reescribe la respuesta corrigiendo solo lo senalado."
    )
    | model
    | parser
)

print("3 cadenas LCEL listas")

3 cadenas LCEL listas


In [ ]:
def self_refine(tarea: str, max_iter: int = 3) -> dict:
    """El loop es Python plano: aqui se ve exactamente cuando para y por que."""
    borrador = generar.invoke({"tarea": tarea})
    historial = []

    for i in range(max_iter):
        critica = criticar.invoke({"tarea": tarea, "borrador": borrador})
        historial.append({"iter": i, "critica": critica[:150]})

        # El criterio de parada lo pone el LLM -> exactamente lo que Kamoi et al. senalan
        if "SIN_PROBLEMAS" in critica:
            return {"salida": borrador, "iteraciones": i, "historial": historial}

        borrador = refinar.invoke(
            {"tarea": tarea, "borrador": borrador, "critica": critica}
        )

    return {"salida": borrador, "iteraciones": max_iter, "historial": historial}


TAREA = (
    "Como explicarías unsando política que es un algoritmo genético"
)

res = self_refine(TAREA)
print(f"Iteraciones: {res['iteraciones']}\n")
print("RESULTADO:\n", res["salida"])
print("\n--- criticas ---")
for h in res["historial"]:
    print(f"  iter {h['iter']}: {h['critica'][:200]}...")

Iteraciones: 1

RESULTADO:
 Un algoritmo genético es una técnica de optimización que imita el proceso de selección natural. Utiliza una población de soluciones potenciales (individuos) que evolucionan a través de operaciones como selección, cruce y mutación, para encontrar la mejor solución posible a un problema específico.

Este algoritmo comienza con una población inicial aleatoria y, en cada generación, selecciona los mejores individuos para cruzarlos y mutarlos, buscando mejorar continuamente las soluciones. Se basa en principios biológicos como la reproducción, la selección natural y la mutación. Es especialmente útil en problemas donde las soluciones son complejas o tienen muchas variables, como en optimización, aprendizaje automático, diseño de ingeniería, entre otros. El objetivo principal es encontrar una solución óptima o casi óptima en un espacio de búsqueda muy grande.

--- criticas ---
  iter 0: La respuesta propuesta es correcta y clara en términos generales, pero puede m

### Experimento 1.1 — ¿la crítica aporta algo?

Baseline **con el mismo presupuesto de llamadas** (punto 3 del checklist de la clase).

In [5]:
mejorar = (
    ChatPromptTemplate.from_template("Mejora este texto:\n\n{borrador}")
    | model
    | parser
)

borrador = generar.invoke({"tarea": TAREA})
for _ in range(res["iteraciones"]):
    borrador = mejorar.invoke({"borrador": borrador})

print("SIN CRITICA (mismo presupuesto):\n", borrador)
print("\n" + "=" * 70 + "\n")
print("CON CRITICA:\n", res["salida"])
print("\n>> La diferencia justifica el doble de llamadas?")

SIN CRITICA (mismo presupuesto):
 Un algoritmo genético es una técnica de optimización inspirada en los procesos de la evolución natural. Funciona mediante una población de soluciones candidatas, que se evalúan utilizando una función de fitness para determinar su calidad. Las soluciones más aptas son seleccionadas para reproducirse mediante operadores como cruce y mutación, generando nuevas soluciones. Este ciclo se repite a lo largo de varias generaciones, con el objetivo de mejorar progresivamente las soluciones hasta alcanzar la mejor posible o cumplir con un criterio de parada establecido.


CON CRITICA:
 Un algoritmo genético es una técnica de optimización que imita el proceso de selección natural. Utiliza una población de soluciones potenciales (individuos) que evolucionan a través de operaciones como selección, cruce y mutación, para encontrar la mejor solución posible a un problema específico.

Este algoritmo comienza con una población inicial aleatoria y, en cada generación, s

---
# Parte 2 — Reflexion con `create_agent` y un verificador real

Aquí LangChain v1 brilla: `create_agent` ya trae el loop de herramientas.

Separamos los dos niveles que vimos en la teoría:

- **Loop interno (intra-tarea)** → lo hace `create_agent`: escribe código, llama a la tool, ve el error, corrige.
- **Loop externo (inter-tarea)** → lo escribimos nosotros: destila una lección y la inyecta en el `system_prompt` de la siguiente tarea.

In [6]:
import io, contextlib, traceback, re
from typing import List
from langchain.tools import tool
from langchain.agents import create_agent

# Estado del ejercicio actual (lo lee la tool)
_TESTS_ACTUALES = {"tests": ""}


@tool
def ejecutar_tests(codigo: str) -> str:
    """Ejecuta el codigo Python contra los tests del problema actual.

    Devuelve 'TODOS LOS TESTS PASARON' o el traceback del error.
    Usa esta herramienta SIEMPRE antes de dar una respuesta final.
    """
    entorno = {}
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            exec(codigo, entorno)
            exec(_TESTS_ACTUALES["tests"], entorno)
        return "TODOS LOS TESTS PASARON"
    except Exception:
        return "FALLO:\n" + traceback.format_exc(limit=2)


# Memoria de lecciones: el feedback INTER-tarea
MEMORIA_LECCIONES: List[str] = []
print("tool y memoria listas")

tool y memoria listas


In [7]:
SYSTEM_BASE = (
    "Eres un programador Python. Escribe la funcion pedida y VERIFICALA "
    "llamando a la herramienta ejecutar_tests antes de responder. "
    "Si falla, corrige y vuelve a verificar. "
    "Cuando pasen los tests, responde solo con el codigo final en un bloque ```python."
)


def construir_system_prompt() -> str:
    if not MEMORIA_LECCIONES:
        return SYSTEM_BASE
    lecciones = "\n".join(f"- {l}" for l in MEMORIA_LECCIONES)
    return f"{SYSTEM_BASE}\n\nLecciones de problemas anteriores:\n{lecciones}"


def extraer_codigo(texto: str) -> str:
    m = re.search(r"```(?:python)?\s*(.*?)```", texto, re.DOTALL)
    return m.group(1).strip() if m else texto.strip()


# La leccion se destila con una cadena LCEL corriente
destilar_leccion = (
    ChatPromptTemplate.from_template(
        "Un agente resolvio este problema:\n{problema}\n\n"
        "Le costo {n} llamadas a herramientas. Traza resumida:\n{traza}\n\n"
        "Escribe UNA sola frase, general y reutilizable, que le habria ahorrado errores. "
        "Prohibido mencionar este problema concreto. Maximo 20 palabras."
    )
    | model
    | parser
)


def resolver(problema: str, tests: str) -> dict:
    """Loop interno: create_agent. Loop externo: nosotros."""
    _TESTS_ACTUALES["tests"] = tests

    agente = create_agent(
        model=model,
        tools=[ejecutar_tests],
        system_prompt=construir_system_prompt(),
    )

    salida = agente.invoke({"messages": [{"role": "user", "content": problema}]})
    mensajes = salida["messages"]

    # Contamos cuantas veces uso el verificador: es la senal externa
    llamadas = sum(1 for m in mensajes if getattr(m, "type", "") == "tool")
    traza = " | ".join(
        m.text[:70] for m in mensajes if getattr(m, "type", "") == "tool"
    )
    codigo = extraer_codigo(mensajes[-1].text)

    entorno = {}
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            exec(codigo, entorno)
            exec(tests, entorno)
        exito = True
    except Exception:
        exito = False

    return {"exito": exito, "llamadas": llamadas, "codigo": codigo, "traza": traza,
            "problema": problema}

print("agente listo")

agente listo


In [8]:
PROBLEMAS = [
    {
        "problema": "Escribe rotar(lista, k) que rote una lista k posiciones a la derecha. "
                    "Debe funcionar con k mayor que len(lista) y con lista vacia.",
        "tests": "assert rotar([1,2,3,4,5], 2) == [4,5,1,2,3]\n"
                 "assert rotar([1,2,3], 7) == [3,1,2]\n"
                 "assert rotar([], 3) == []",
    },
    {
        "problema": "Escribe media_movil(xs, n) que devuelva la media movil de ventana n. "
                    "Si n es mayor que len(xs), devuelve lista vacia.",
        "tests": "assert media_movil([1,2,3,4], 2) == [1.5, 2.5, 3.5]\n"
                 "assert media_movil([1,2], 5) == []",
    },
    {
        "problema": "Escribe agrupar(pares) que reciba lista de tuplas (clave, valor) "
                    "y devuelva un dict clave -> lista de valores, preservando el orden.",
        "tests": "assert agrupar([('a',1),('b',2),('a',3)]) == {'a':[1,3], 'b':[2]}\n"
                 "assert agrupar([]) == {}",
    },
]

MEMORIA_LECCIONES.clear()
resultados = []

for i, p in enumerate(PROBLEMAS, 1):
    r = resolver(p["problema"], p["tests"])
    resultados.append(r)
    print(f"Problema {i}: {'OK' if r['exito'] else 'FALLO'} "
          f"({r['llamadas']} llamadas al verificador)")

    # Loop externo: destilar y guardar
    if r["llamadas"] > 1:
        leccion = destilar_leccion.invoke(
            {"problema": p["problema"], "n": r["llamadas"], "traza": r["traza"]}
        ).strip().lstrip("-* ")
        if leccion not in MEMORIA_LECCIONES:
            MEMORIA_LECCIONES.append(leccion)
            print(f"    leccion: {leccion}")

print("\n=== MEMORIA ACUMULADA ===")
for l in MEMORIA_LECCIONES:
    print(" -", l)

Problema 1: OK (1 llamadas al verificador)
Problema 2: OK (1 llamadas al verificador)
Problema 3: OK (1 llamadas al verificador)

=== MEMORIA ACUMULADA ===


### Experimento 2.1 — ¿la memoria inter-tarea reduce el esfuerzo?

Medimos **llamadas al verificador**: menos llamadas = acertó antes.

In [9]:
con_memoria = sum(r["llamadas"] for r in resultados)

MEMORIA_LECCIONES.clear()
sin_memoria = 0
for p in PROBLEMAS:
    r = resolver(p["problema"], p["tests"])
    sin_memoria += r["llamadas"]
    MEMORIA_LECCIONES.clear()  # se borra tras cada problema -> sin transferencia

print(f"Llamadas al verificador CON memoria inter-tarea: {con_memoria}")
print(f"Llamadas al verificador SIN memoria inter-tarea: {sin_memoria}")
print("\n>> Con 3 problemas el ruido es alto. Cuantos harian falta para creer la diferencia?")

Llamadas al verificador CON memoria inter-tarea: 3
Llamadas al verificador SIN memoria inter-tarea: 4

>> Con 3 problemas el ruido es alto. Cuantos harian falta para creer la diferencia?


---
## Experimento 3 — Reproducir el hallazgo de Kamoi et al.

Quitamos el verificador y dejamos que **el LLM juzgue su propio código**.

> Predicción: sin señal verificable, el juicio deja de ser fiable — y un loop que optimiza contra un juez roto optimiza hacia el lugar equivocado.

In [10]:
juez_llm = (
    ChatPromptTemplate.from_template(
        "Codigo:\n```python\n{codigo}\n```\n\n"
        "Tests que debe pasar:\n{tests}\n\n"
        "Sin ejecutar nada, responde solo PASA o FALLA."
    )
    | model
    | parser
)


def ejecutar_de_verdad(codigo: str, tests: str) -> bool:
    entorno = {}
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            exec(codigo, entorno)
            exec(tests, entorno)
        return True
    except Exception:
        return False


print(f"{'problema':<10} {'LLM dice':<12} {'realidad':<12} {'coincide'}")
print("-" * 48)
aciertos = 0
for i, (p, r) in enumerate(zip(PROBLEMAS, resultados), 1):
    cree = juez_llm.invoke({"codigo": r["codigo"], "tests": p["tests"]})
    cree = cree.strip().upper().startswith("PASA")
    real = ejecutar_de_verdad(r["codigo"], p["tests"])
    ok = cree == real
    aciertos += ok
    print(f"{i:<10} {str(cree):<12} {str(real):<12} {'si' if ok else 'NO'}")

print(f"\nEl LLM acerto en {aciertos}/{len(PROBLEMAS)} juicios.")
print("\n>> Recuerda la Tabla 2 de SCoRe: en MATH, pedirle a Gemini 1.5 Flash que")
print(">> se revise sin entrenamiento le hace PERDER 11 puntos (delta = -11.2%).")

problema   LLM dice     realidad     coincide
------------------------------------------------
1          True         True         si
2          True         True         si
3          True         True         si

El LLM acerto en 3/3 juicios.

>> Recuerda la Tabla 2 de SCoRe: en MATH, pedirle a Gemini 1.5 Flash que
>> se revise sin entrenamiento le hace PERDER 11 puntos (delta = -11.2%).
